# DeepLearning.ai - [Knowledge Graphs for RAG](https://www.deeplearning.ai/courses/knowledge-graphs-rag)

## Lesson 2: Querying Knowledge Graphs with Cypher

### Import packages and set up Neo4


In [35]:
from dotenv import load_dotenv
import hashlib
import os
import json
import neo4j
import textwrap

from langchain_community.graphs import Neo4jGraph

# Warning control
import warnings
warnings.filterwarnings("ignore")

In [36]:
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

print(NEO4J_URI)
print(NEO4J_DATABASE)
print(NEO4J_USERNAME)
print(hashlib.sha256(NEO4J_PASSWORD.encode()).hexdigest()[:8])
# printf '%s' "$NEO4J_PASSWORD" | sha256sum | cut -c1-8

bolt://172.30.2.177:7687
neo4j
neo4j
b90caf5c


In [37]:
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

Get setup to use [Titan embeddings via Amazon Bedrock endpoint](https://neo4j.com/docs/genai/plugin/current/embeddings/):

In [38]:
import boto3

frozen = boto3.Session().get_credentials().get_frozen_credentials()   # default chain: ~/.aws
print("key id prefix:", frozen.access_key[:4])          # AKIA = long-lived, ASIA = temporary
print("session token present:", frozen.token is not None)

key id prefix: AKIA
session token present: False


In [39]:
aws_cfg = {
    "accessKeyId": frozen.access_key,
    "secretAccessKey": frozen.secret_key,
    "region": "us-east-1",
    "model": "amazon.titan-embed-text-v2:0",
}

kg.query("""
RETURN vector_dimension_count(ai.text.embed($text, 'bedrock-titan', $config)) AS dimensions
""", params={"text": "Welcome to the Real World", "config": aws_cfg})

[{'dimensions': 1024}]

### Querying the movie knowledge graph 
- Match all nodes in the graph

In [9]:
cypher = """
  MATCH (n) 
  RETURN count(n)
  """

In [10]:
result = kg.query(cypher)
result

[{'count(n)': 209}]

In [11]:
cypher = """
  MATCH (n) 
  RETURN count(n) AS numberOfNodes
  """

In [12]:
result = kg.query(cypher)
result

[{'numberOfNodes': 209}]

In [13]:
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 209 nodes in this graph.


- Match only the Movie nodes by specifying the node label

In [14]:
cypher = """
  MATCH (n:Movie) 
  RETURN count(n) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 39}]

- Change the variable name in the node pattern match for improved readability

In [15]:
cypher = """
  MATCH (m:Movie) 
  RETURN count(m) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 39}]

- Match only the Person nodes

In [16]:
cypher = """
  MATCH (people:Person) 
  RETURN count(people) AS numberOfPeople
  """
kg.query(cypher)

[{'numberOfPeople': 134}]

- Match a single person by specifying the value of the name property on the Person node

In [17]:
cypher = """
  MATCH (tom:Person {name:"Tom Hanks"}) 
  RETURN tom
  """
kg.query(cypher)

[{'tom': {'born': 1956, 'name': 'Tom Hanks'}}]

- Match a single Movie by specifying the value of the title property

In [18]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas
  """
kg.query(cypher)

[{'cloudAtlas': {'taglineEmbedding': [-0.06823286414146423,
    0.01721089519560337,
    0.007717038970440626,
    -0.04129813238978386,
    0.043067317456007004,
    -0.015023749321699142,
    0.032688867300748825,
    -0.01901252381503582,
    0.03367938473820686,
    0.01665552146732807,
    -0.012972106225788593,
    0.032820556312799454,
    -0.011295485310256481,
    -0.004473534878343344,
    -0.0018512496026232839,
    0.02050497569143772,
    0.0015287123387679458,
    -0.038172002881765366,
    0.028112275525927544,
    0.041868776082992554,
    -0.05517296493053436,
    0.04985586926341057,
    -0.008158857934176922,
    0.01181364431977272,
    -0.003820826532319188,
    -0.05157161504030228,
    0.04802561178803444,
    0.027449069544672966,
    0.07864566147327423,
    0.023722713813185692,
    0.012687739916145802,
    0.037379972636699677,
    -0.038322776556015015,
    0.015390181913971901,
    0.06946957111358643,
    0.10094272345304489,
    0.002503958297893405,
   

- Return only the released property of the matched Movie node

In [19]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released
  """
kg.query(cypher)

[{'cloudAtlas.released': 2012}]

- Return two properties

In [20]:
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released, cloudAtlas.tagline
  """
kg.query(cypher)

[{'cloudAtlas.released': 2012,
  'cloudAtlas.tagline': 'Everything is connected'}]

### Cypher patterns with conditional matching

In [21]:
cypher = """
  MATCH (nineties:Movie) 
  WHERE nineties.released >= 1990 
    AND nineties.released < 2000 
  RETURN nineties.title
  """
kg.query(cypher)

[{'nineties.title': 'Joe Versus the Volcano'},
 {'nineties.title': 'A Few Good Men'},
 {'nineties.title': 'Unforgiven'},
 {'nineties.title': 'Hoffa'},
 {'nineties.title': 'A League of Their Own'},
 {'nineties.title': 'Sleepless in Seattle'},
 {'nineties.title': 'Johnny Mnemonic'},
 {'nineties.title': 'Apollo 13'},
 {'nineties.title': 'That Thing You Do'},
 {'nineties.title': 'The Birdcage'},
 {'nineties.title': 'Twister'},
 {'nineties.title': "The Devil's Advocate"},
 {'nineties.title': 'As Good as It Gets'},
 {'nineties.title': 'What Dreams May Come'},
 {'nineties.title': "You've Got Mail"},
 {'nineties.title': 'When Harry Met Sally'},
 {'nineties.title': 'The Matrix'},
 {'nineties.title': 'Snow Falling on Cedars'},
 {'nineties.title': 'The Green Mile'},
 {'nineties.title': 'Bicentennial Man'}]

### Pattern matching with multiple nodes

In [22]:
cypher = """
  MATCH (actor:Person)-[:ACTED_IN]->(movie:Movie) 
  RETURN actor.name, movie.title LIMIT 10
  """
kg.query(cypher)

[{'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix'},
 {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix'},
 {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix'},
 {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix Reloaded'},
 {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Revolutions'},
 {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix Revolutions'}]

In [23]:
cypher = """
  MATCH (tom:Person {name: "Tom Hanks"})-[:ACTED_IN]->(t:Movie) 
  RETURN tom.name,t.title LIMIT 10
  """
kg.query(cypher)

[{'tom.name': 'Tom Hanks', 't.title': 'A League of Their Own'},
 {'tom.name': 'Tom Hanks', 't.title': 'The Polar Express'},
 {'tom.name': 'Tom Hanks', 't.title': "Charlie Wilson's War"},
 {'tom.name': 'Tom Hanks', 't.title': 'Cast Away'},
 {'tom.name': 'Tom Hanks', 't.title': 'Apollo 13'},
 {'tom.name': 'Tom Hanks', 't.title': 'The Green Mile'},
 {'tom.name': 'Tom Hanks', 't.title': 'The Da Vinci Code'},
 {'tom.name': 'Tom Hanks', 't.title': 'Cloud Atlas'},
 {'tom.name': 'Tom Hanks', 't.title': 'That Thing You Do'},
 {'tom.name': 'Tom Hanks', 't.title': 'Joe Versus the Volcano'}]

In [24]:
cypher = """
  MATCH (tom:Person {name:"Tom Hanks"})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors) 
  RETURN coActors.name, m.title LIMIT 10
  """
kg.query(cypher)

[{'coActors.name': 'Geena Davis', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Bill Paxton', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Madonna', 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Lori Petty', 'm.title': 'A League of Their Own'},
 {'coActors.name': "Rosie O'Donnell", 'm.title': 'A League of Their Own'},
 {'coActors.name': 'Philip Seymour Hoffman',
  'm.title': "Charlie Wilson's War"},
 {'coActors.name': 'Julia Roberts', 'm.title': "Charlie Wilson's War"},
 {'coActors.name': 'Helen Hunt', 'm.title': 'Cast Away'},
 {'coActors.name': 'Ed Harris', 'm.title': 'Apollo 13'},
 {'coActors.name': 'Gary Sinise', 'm.title': 'Apollo 13'}]

### Create a vector index 

In [25]:
kg.query("""
  CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS
  FOR (m:Movie) ON (m.taglineEmbedding)
  OPTIONS { indexConfig: {
    `vector.dimensions`: 1024,
    `vector.similarity_function`: 'cosine'
  }}"""
)

# kg.query("DROP INDEX movie_tagline_embeddings IF EXISTS")

[]

In [26]:
kg.query("""
    SHOW VECTOR INDEXES
        YIELD *
        ORDER BY id
    """
)

[{'id': 9,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-2026.08',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2026, 10, 7, 17, 24, 27, 534000000, tzinfo=<UTC>),
  'readCount': 12,
  'trackedSince': neo4j.time.DateTime(2026, 10, 5, 17, 46, 42, 366000000, tzinfo=<UTC>),
  'options': {'indexConfig': {'vector.dimensions': 1024,
    'vector.default_search_expansion_factor': 3.0,
    'vector.hnsw.m': 16,
    'vector.quantization.type': 'BINARY',
    'vector.similarity_function': 'COSINE',
    'vector.hnsw.ef_construction': 100}},
  'failureMessage': '',
  'createStatement': "CREATE VECTOR INDEX `movie_tagline_embeddings` FOR (n:`Movie`) ON (n.`taglineEmbedding`) OPTIONS {indexConfig: {`vector.default_search_expansion_factor`: 3.0,`vector.dimensions`: 1024,`vector.hnsw.ef_construction`: 100,`vec

In [27]:
kg.query("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")

[{'name': 'Neo4j Kernel', 'versions': ['2026.09.0'], 'edition': 'community'},
 {'name': 'Cypher', 'versions': ['5', '25'], 'edition': ''}]

In [28]:
kg.query("""
  SHOW FUNCTIONS YIELD name
  WHERE name STARTS WITH 'genai' OR name STARTS WITH 'ai.'
  RETURN name
""")

[{'name': 'ai.image.embed'},
 {'name': 'ai.text.aggregateCompletion'},
 {'name': 'ai.text.aggregateStructuredCompletion'},
 {'name': 'ai.text.chat'},
 {'name': 'ai.text.chunkByTokenLimit'},
 {'name': 'ai.text.completion'},
 {'name': 'ai.text.embed'},
 {'name': 'ai.text.structuredCompletion'},
 {'name': 'ai.text.tokenCount'},
 {'name': 'genai.vector.encode'}]

In [29]:
kg.query("""
  CALL genai.vector.listEncodingProviders()
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.listEncodingProviders is deprecated. It is replaced by ai.text.embed.providers.', position=<SummaryInputPosition line=2, column=3, offset=3>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 3, 'line': 2, 'column': 3}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n  CALL genai.vector.listEncodingProviders()\n'


[{'name': 'AzureOpenAI',
  'requiredConfigType': '{ token :: STRING NOT NULL, resource :: STRING NOT NULL, deployment :: STRING NOT NULL }',
  'optionalConfigType': '{ dimensions :: INTEGER }',
  'defaultConfig': {}},
 {'name': 'Bedrock',
  'requiredConfigType': '{ accessKeyId :: STRING NOT NULL, secretAccessKey :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, region :: STRING NOT NULL, dimensions :: INTEGER, normalize :: BOOLEAN }',
  'defaultConfig': {'region': 'us-east-1',
   'model': 'amazon.titan-embed-text-v1'}},
 {'name': 'OpenAI',
  'requiredConfigType': '{ token :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, dimensions :: INTEGER }',
  'defaultConfig': {'model': 'text-embedding-ada-002'}},
 {'name': 'VertexAI',
  'requiredConfigType': '{ token :: STRING NOT NULL, projectId :: STRING NOT NULL }',
  'optionalConfigType': '{ model :: STRING NOT NULL, region :: STRING NOT NULL, taskType :: STRING, title :: STRING, autoTruncat

In [30]:
kg.query("""
  SHOW PROCEDURES YIELD name
  WHERE name STARTS WITH 'genai' OR name = 'db.create.setNodeVectorProperty'
  RETURN name
""")

[{'name': 'db.create.setNodeVectorProperty'},
 {'name': 'genai.vector.encodeBatch'},
 {'name': 'genai.vector.listEncodingProviders'}]

### Populate the vector index
- Calculate vector representation for each movie tagline using OpenAI
- Add vector to the `Movie` node as `taglineEmbedding` property


In [31]:
kg.query("""
    MATCH (movie:Movie) WHERE movie.tagline IS NOT NULL
    WITH movie, ai.text.embed(
        movie.tagline,
        'bedrock-titan',
        $config) AS vector
    CALL db.create.setNodeVectorProperty(movie, 'taglineEmbedding', vector)
    """,
    params={"config": aws_cfg}
    )

[]

In [32]:
result = kg.query("""
    MATCH (m:Movie) 
    WHERE m.tagline IS NOT NULL
    RETURN m.tagline, m.taglineEmbedding
    LIMIT 1
    """
)

In [33]:
result[0]['m.tagline']

'Welcome to the Real World'

In [34]:
len(result[0]['m.taglineEmbedding'])

result[0]['m.taglineEmbedding'][:10]

[-0.038327254354953766,
 0.07644597440958023,
 0.022722776979207993,
 -0.028067199513316154,
 -0.007544337771832943,
 -0.013895501382648945,
 0.019532693549990654,
 -0.003534639021381736,
 0.002237200504168868,
 0.03942720964550972]

### Similarity search
- Calculate embedding for question
- Identify matching movies based on similarity of question and `taglineEmbedding` vectors

In [35]:
question = "What movies are about love?"

kg.query("""
    WITH ai.text.embed($question, 'bedrock-titan', $config) AS question_embedding
    MATCH (movie:Movie)
    SEARCH movie IN (
        VECTOR INDEX movie_tagline_embeddings
        FOR question_embedding
        LIMIT $top_k
    ) SCORE AS score
    RETURN movie.title, movie.tagline, score
    ORDER BY score DESC
    """,
    params={"config": aws_cfg,
            "question": question,
            "top_k": 5
            }
    )

[{'movie.title': 'Joe Versus the Volcano',
  'movie.tagline': 'A story of love, lava and burning desire.',
  'score': 0.6089553236961365},
 {'movie.title': 'Snow Falling on Cedars',
  'movie.tagline': 'First loves last. Forever.',
  'score': 0.5974831581115723},
 {'movie.title': "You've Got Mail",
  'movie.tagline': 'At odds in life... in love on-line.',
  'score': 0.5863548517227173},
 {'movie.title': 'Stand By Me',
  'movie.tagline': "For some, it's the last real taste of innocence, and the first real taste of life. But for everyone, it's the time that memories are made of.",
  'score': 0.5583323836326599},
 {'movie.title': 'As Good as It Gets',
  'movie.tagline': 'A comedy from the heart that goes for the throat.',
  'score': 0.5537737011909485}]

In [36]:
question = "What movies are about freedom?"

kg.query("""
    WITH ai.text.embed($question, 'bedrock-titan', $config) AS question_embedding
    MATCH (movie:Movie)
    SEARCH movie IN (
        VECTOR INDEX movie_tagline_embeddings
        FOR question_embedding
        LIMIT $top_k
    ) SCORE AS score
    RETURN movie.title, movie.tagline, score
    ORDER BY score DESC
    """,
    params={"config": aws_cfg,
            "question": question,
            "top_k": 5
            }
    )

[{'movie.title': 'RescueDawn',
  'movie.tagline': "Based on the extraordinary true story of one man's fight for freedom",
  'score': 0.6479344964027405},
 {'movie.title': 'V for Vendetta',
  'movie.tagline': 'Freedom! Forever!',
  'score': 0.6410220861434937},
 {'movie.title': 'The Matrix Reloaded',
  'movie.tagline': 'Free your mind',
  'score': 0.6005020141601562},
 {'movie.title': 'The Da Vinci Code',
  'movie.tagline': 'Break The Codes',
  'score': 0.5902088284492493},
 {'movie.title': 'Speed Racer',
  'movie.tagline': 'Speed has no limits',
  'score': 0.579997181892395}]

# Lesson 4: Constructing a Knowledge Graph from Text Documents

### Import packages and set up Neo4j

In [37]:
# Common data processing
import json
import textwrap

# Langchain
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_openai import ChatOpenAI

In [38]:
# Global constants
VECTOR_INDEX_NAME = 'form_10k_chunks'
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'

### Take a look at a Form 10-K json file

- Publicly traded companies are required to fill a form 10-K each year with the Securities and Exchange Commision (SEC)
- You can search these filings using the SEC's [EDGAR database](https://www.sec.gov/edgar/search/)
- For the next few lessons, you'll work with a single 10-K form for a company called [NetApp](https://www.netapp.com/)

In [39]:
first_file_name = "./data/form10k/0000950170-23-027948.json"
first_file_as_object = json.load(open(first_file_name))
type(first_file_as_object)
for k,v in first_file_as_object.items():
    print(k, type(v))

item1 <class 'str'>
item1a <class 'str'>
item7 <class 'str'>
item7a <class 'str'>
cik <class 'str'>
cusip6 <class 'str'>
cusip <class 'list'>
names <class 'list'>
source <class 'str'>


In [40]:
item1_text = first_file_as_object['item1']
item1_text[0:1500]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

### Split Form 10-K sections into chunks
- Set up text splitter using LangChain
- For now, split only the text from the "item 1" section 

In [41]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap  = 200,
    length_function = len,
    is_separator_regex = False,
)

item1_text_chunks = text_splitter.split_text(item1_text)

type(item1_text_chunks)
# list

len(item1_text_chunks)

254

In [42]:
item1_text_chunks[0]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

- Set up helper function to chunk all sections of the Form 10-K
- You'll limit the number of chunks in each section to 20 to speed things up

In [43]:
def split_form10k_data_from_file(file):
    chunks_with_metadata = [] # use this to accumlate chunk records
    file_as_object = json.load(open(file)) # open the json file
    for item in ['item1','item1a','item7','item7a']: # pull these keys from the json
        print(f'Processing {item} from {file}') 
        item_text = file_as_object[item] # grab the text of the item
        item_text_chunks = text_splitter.split_text(item_text) # split the text into chunks
        chunk_seq_id = 0
        for chunk in item_text_chunks[:20]: # only take the first 20 chunks
            form_id = file[file.rindex('/') + 1:file.rindex('.')] # extract form id from file name
            # finally, construct a record with metadata and the chunk text
            chunks_with_metadata.append({
                'text': chunk, 
                # metadata from looping...
                'f10kItem': item,
                'chunkSeqId': chunk_seq_id,
                # constructed metadata...
                'formId': f'{form_id}', # pulled from the filename
                'chunkId': f'{form_id}-{item}-chunk{chunk_seq_id:04d}',
                # metadata from file...
                'names': file_as_object['names'],
                'cik': file_as_object['cik'],
                'cusip6': file_as_object['cusip6'],
                'source': file_as_object['source'],
            })
            chunk_seq_id += 1
        print(f'\tSplit into {chunk_seq_id} chunks')
    return chunks_with_metadata

In [44]:
first_file_chunks = split_form10k_data_from_file(first_file_name)

Processing item1 from ./data/form10k/0000950170-23-027948.json
	Split into 20 chunks
Processing item1a from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks
Processing item7 from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks
Processing item7a from ./data/form10k/0000950170-23-027948.json
	Split into 1 chunks


In [45]:
first_file_chunks[0]

{'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved clou

### Create graph nodes using text chunks

In [46]:
merge_chunk_node_query = """
MERGE(mergedChunk:Chunk {chunkId: $chunkParam.chunkId})
    ON CREATE SET 
        mergedChunk.names = $chunkParam.names,
        mergedChunk.formId = $chunkParam.formId, 
        mergedChunk.cik = $chunkParam.cik, 
        mergedChunk.cusip6 = $chunkParam.cusip6, 
        mergedChunk.source = $chunkParam.source, 
        mergedChunk.f10kItem = $chunkParam.f10kItem, 
        mergedChunk.chunkSeqId = $chunkParam.chunkSeqId, 
        mergedChunk.text = $chunkParam.text
RETURN mergedChunk
"""

- Set up connection to graph instance using LangChain

In [47]:
kg.query(merge_chunk_node_query, 
         params={'chunkParam':first_file_chunks[0]})

[{'mergedChunk': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'names': ['Netapp Inc', 'NETAPP INC'],
   'textEmbedding': [0.012315970845520496,
    0.0564827136695385,
    -0.016076290979981422,
    -0.04092215374112129,
    0.016637828201055527,
    -0.014634944498538971,
    0.04813723638653755,
    0.026221899315714836,
    -0.007163951173424721,
    0.015501960180699825,
    0.0010919076157733798,
    0.026420975103974342,
    -0.036000579595565796,
    0.011353936046361923,
    -0.007913239300251007,
    -0.012589824385941029,
    0.005533285439014435,
    0.06880379468202591,
    -0.0007723713060840964,
    -0.0015071264933794737,
    0.012203353457152843,
    0.004162668250501156,
    0.06309783458709717,
    0.013698234222829342,
    -0.037816502153873444,
    -0.011297168210148811,
    0.0006724249687977135,
    0.016592765226960182,
    0.07959188520908356,
    -0.032358746975660324,
    -0.010658122599124908,
    0.029213137924671173,
    0.03787208721041679

- Create a uniqueness constraint to avoid duplicate chunks

In [48]:
kg.query("""
CREATE CONSTRAINT unique_chunk IF NOT EXISTS 
    FOR (c:Chunk) REQUIRE c.chunkId IS UNIQUE
""")


[]

In [49]:
kg.query("""
    SHOW INDEXES
        YIELD *
        ORDER BY id
    """)

[{'id': 1,
  'name': 'index_460996c0',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2026, 10, 7, 19, 15, 12, 124000000, tzinfo=<UTC>),
  'readCount': 279,
  'trackedSince': neo4j.time.DateTime(2026, 10, 2, 1, 25, 14, 616000000, tzinfo=<UTC>),
  'options': {'indexConfig': {}},
  'failureMessage': '',
  'createStatement': 'CREATE LOOKUP INDEX `index_460996c0` FOR (n) ON EACH labels(n)'},
 {'id': 2,
  'name': 'index_1b9dcc97',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0,
  'trackedSince': neo4j.time.DateTime(2026, 10, 2, 1, 25, 14, 640000000, tzinfo=<UTC>),
  'options': {'indexConfig': {}

- Loop through and create nodes for all chunks
- Should create 23 nodes because you set a limit of 20 chunks in the text splitting function above

In [50]:
node_count = 0
for chunk in first_file_chunks:
    print(f"Creating `:Chunk` node for chunk ID {chunk['chunkId']}")
    kg.query(merge_chunk_node_query, 
            params={
                'chunkParam': chunk
            })
    node_count += 1
print(f"Created {node_count} nodes")

Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0000
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0001
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0002
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0003
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0004
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0005
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0006
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0007
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0008
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0009
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0010
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0011
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0012
Creating `:Chunk` node for chunk ID 0000950170-23-0

In [94]:
kg.query("""
    MATCH (n:Chunk)
    RETURN count(n) as nodeCount
         """)

[{'nodeCount': 23}]

### Create a vector index

In [52]:
kg.query("""
  CREATE VECTOR INDEX `form_10k_chunks` IF NOT EXISTS
  FOR (c:Chunk) ON (c.textEmbedding)
  OPTIONS { indexConfig: {
    `vector.dimensions`: 1024,
    `vector.similarity_function`: 'cosine'
  }}"""
)

# kg.query("DROP INDEX `form_10k_chunks` IF EXISTS")

[]

In [53]:
kg.query("""
    SHOW INDEXES
        YIELD *
        ORDER BY id
    """)

[{'id': 1,
  'name': 'index_460996c0',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2026, 10, 7, 19, 15, 12, 124000000, tzinfo=<UTC>),
  'readCount': 279,
  'trackedSince': neo4j.time.DateTime(2026, 10, 2, 1, 25, 14, 616000000, tzinfo=<UTC>),
  'options': {'indexConfig': {}},
  'failureMessage': '',
  'createStatement': 'CREATE LOOKUP INDEX `index_460996c0` FOR (n) ON EACH labels(n)'},
 {'id': 2,
  'name': 'index_1b9dcc97',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0,
  'trackedSince': neo4j.time.DateTime(2026, 10, 2, 1, 25, 14, 640000000, tzinfo=<UTC>),
  'options': {'indexConfig': {}

### Calculate embedding vectors for chunks and populate index
- This query calculates the embedding vector and stores it as a property called `textEmbedding` on each `Chunk` node.

In [54]:
kg.query("""
    MATCH (chunk:Chunk) WHERE chunk.textEmbedding IS NULL
    WITH chunk, ai.text.embed(
        chunk.text,
        'bedrock-titan',
        $config) AS vector
    CALL db.create.setNodeVectorProperty(chunk, 'textEmbedding', vector)
    """,
    params={"config": aws_cfg}
    )

[]

In [55]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Movie {title: STRING, taglineEmbedding: LIST, released: INTEGER, tagline: STRING}
Person {born: INTEGER, name: STRING}
Classification {name: STRING, ers_2025: STRING, family: STRING, ers_2015: STRING}
Chunk {textEmbedding: LIST, f10kItem: STRING, chunkSeqId: INTEGER, text: STRING, cik: STRING, cusip6: STRING, names: LIST, formId: STRING, source: STRING, chunkId: STRING}
Form {cik: STRING, cusip6: STRING, names: LIST, formId: STRING, source: STRING}
Relationship properties:
ACTED_IN {roles: LIST}
REVIEWED {summary: STRING, rating: INTEGER}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person)-[:REVIEWED]->(:Movie)


In [56]:
result = kg.query("""
    MATCH (c:Chunk) 
    WHERE c.text IS NOT NULL
    RETURN c.text, c.textEmbedding
    LIMIT 1
    """
)

if len(result[0]['c.textEmbedding']) == 1024:
    print(result[0]['c.textEmbedding'][:10])

[0.012315970845520496, 0.0564827136695385, -0.016076290979981422, -0.04092215374112129, 0.016637828201055527, -0.014634944498538971, 0.04813723638653755, 0.026221899315714836, -0.007163951173424721, 0.015501960180699825]


### Use similarity search to find relevant chunks

- Setup a help function to perform similarity search using the vector index

In [57]:
def neo4j_vector_search(question):
    vector_search_results = kg.query("""
        WITH ai.text.embed($question, 'bedrock-titan', $config) AS question_embedding
        MATCH (chunk:Chunk)
        SEARCH chunk IN (
            VECTOR INDEX form_10k_chunks
            FOR question_embedding
            LIMIT $top_k
        ) SCORE AS score
        RETURN chunk.chunkId, chunk.names, chunk.text, score
        ORDER BY score DESC
        """,
        params={"config": aws_cfg,
            "question": question,
            "top_k": 10
            })
    return vector_search_results

- Ask a question!

In [58]:
search_results = neo4j_vector_search(
    'In a single sentence, tell me about Netapp.'
)

chunkId, names, text, score = (search_results[0][k] for k in ('chunk.chunkId', 'chunk.names', 'chunk.text', 'score'))

print(json.dumps({
    "chunkId": chunkId,
    "names": names,
    "text": text[:40],
    "score": score,
}, indent=2))


{
  "chunkId": "0000950170-23-027948-item1-chunk0000",
  "names": [
    "Netapp Inc",
    "NETAPP INC"
  ],
  "text": ">Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp,",
  "score": 0.8436089754104614
}


### Set up a LangChain RAG workflow to chat with the form

In [59]:
from langchain_aws import BedrockEmbeddings
from langchain_neo4j import Neo4jVector

neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=BedrockEmbeddings(
        model_id="amazon.titan-embed-text-v2:0",
        region_name="us-east-1",
    ),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

retriever = neo4j_vector_store.as_retriever()

- Set up a RetrievalQAWithSourcesChain to carry out question answering
- You can check out the LangChain documentation for this chain [here](https://api.python.langchain.com/en/latest/chains/langchain.chains.qa_with_sources.retrieval.RetrievalQAWithSourcesChain.html)

In [60]:
# chain = RetrievalQAWithSourcesChain.from_chain_type(
#     ChatOpenAI(temperature=0), 
#     chain_type="stuff", 
#     retriever=retriever
# )

"""Replacement for RetrievalQAWithSourcesChain.from_chain_type(llm, chain_type="stuff", retriever=retriever)."""

import logging
import re

from langchain_aws import ChatBedrockConverse
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

logger = logging.getLogger(__name__)

PROMPT = ChatPromptTemplate.from_template(
    'Given the following extracted parts of a long document and a question, create a final answer with references ("SOURCES").\n'
    "If you don't know the answer, just say that you don't know. Don't try to make up an answer.\n"
    'ALWAYS return a "SOURCES" part in your answer, listing the Source of each part you used.\n\n'
    "QUESTION: {question}\n"
    "=========\n"
    "{summaries}\n"
    "=========\n"
    "FINAL ANSWER:"
)


def format_docs(docs, source_key, source_default):
    # chain_type="stuff": every retrieved document goes into one prompt, the model is called once
    blocks = []
    for d in docs:
        if source_key not in d.metadata:
            logger.warning("Retrieved passage has no `%s` metadata; using %r", source_key, source_default)
        blocks.append(f"Content: {d.page_content}\nSource: {d.metadata.get(source_key, source_default)}")
    return "\n\n".join(blocks)


def split_sources(text):
    # Same split as the legacy chain: text before "SOURCES:" is the answer,
    # the first line after it is the sources
    answer, sources = text, ""
    if re.search(r"SOURCES?:", text, re.IGNORECASE):
        answer, sources = re.split(r"SOURCES?:|QUESTION:\s", text, flags=re.IGNORECASE)[:2]
        sources = sources.split("\n")[0].strip()
    return {"answer": answer.strip(), "sources": sources}


def qa_with_sources_chain(llm, retriever, *, source_key="source", source_default="unknown"):
    """Retrieve, put all passages in one prompt, ask the model once and split out its SOURCES.

    Invoke with {"question": ...}; returns {"question": ..., "answer": ..., "sources": ...}.
    """
    answer_and_sources = (
        RunnablePassthrough.assign(summaries=lambda x: format_docs(x["docs"], source_key, source_default))
        | PROMPT
        | llm
        | StrOutputParser()
        | split_sources
    )
    return (
        RunnableParallel(question=lambda x: x["question"], docs=lambda x: retriever.invoke(x["question"]))
        | RunnableParallel(question=lambda x: x["question"], result=answer_and_sources)
        | (lambda x: {"question": x["question"], **x["result"]})
    )

chain = qa_with_sources_chain(
    ChatBedrockConverse(model="us.anthropic.claude-sonnet-5", region_name="us-east-1", max_tokens=8192),
    retriever=retriever,
    source_key="source"
)

In [61]:
chain.invoke({"question": "What is Netapp's primary business?"})

{'question': "What is Netapp's primary business?",
 'answer': "NetApp, Inc. is a global cloud-led, data-centric software company. It gives customers the freedom to manage applications and data across hybrid multicloud environments through its portfolio of cloud services and storage infrastructure, powered by intelligent data management software. This enables applications to run faster, more reliably, and more securely, at a lower cost. NetApp's operations are organized into two segments: Hybrid Cloud and Public Cloud, and it focuses primarily on the enterprise storage and data management, cloud storage, and cloud operations markets.",
 'sources': 'https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm'}

In [62]:
def prettychain(question: str) -> None:
    """Pretty print the chain's response to a question"""
    response = chain.invoke({"question": question})
    print(response["answer"])
    print(f"\nSources: {response['sources'] or 'none cited'}")

In [63]:
question = "What is Netapp's primary business?"
prettychain(question)

NetApp's primary business is as a global cloud-led, data-centric software company. NetApp provides customers the ability to manage applications and data across hybrid multicloud environments through its portfolio of cloud services and storage infrastructure, powered by intelligent data management software. These offerings enable applications to run faster, more reliably, and more securely, at a lower cost. The company's operations are organized into two segments: Hybrid Cloud and Public Cloud. NetApp focuses primarily on the enterprise storage and data management, cloud storage, and cloud operations markets, serving diverse industries such as energy, financial services, government, technology, life sciences, healthcare, and telecommunications.

Sources: https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm


In [64]:
prettychain("Where is Netapp headquartered?")

NetApp is headquartered in San Jose, California.

Sources: https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm


In [65]:
prettychain("""
    Tell me about Netapp. 
    Limit your answer to a single sentence.
""")

NetApp, Inc. is a global cloud-led, data-centric software company, incorporated in 1992 and headquartered in San Jose, California, that provides cloud services and storage infrastructure powered by intelligent data management software to help customers manage applications and data across hybrid multicloud environments.

Sources: https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm


In [66]:
prettychain("""
    Tell me about Apple. 
    Limit your answer to a single sentence.
""")

I don't know.

Sources: None - the provided documents discuss NetApp, Inc., not Apple.


In [67]:
prettychain("""
    Tell me about Apple. 
    Limit your answer to a single sentence.
    If you are unsure about the answer, say you don't know.
""")

I don't know. The provided sources discuss NetApp, Inc., not Apple.

Sources: None


## Lesson 5: Adding Relationships to the SEC Knowledge Graph

### Create a Form 10-K node
- Create a node to represent the entire Form 10-K
- Populate with metadata taken from a single chunk of the form

In [68]:
cypher = """
  MATCH (anyChunk:Chunk) 
  WITH anyChunk LIMIT 1
  RETURN anyChunk { .names, .source, .formId, .cik, .cusip6 } as formInfo
"""
form_info_list = kg.query(cypher)

form_info_list

[{'formInfo': {'cusip6': '64110D',
   'formId': '0000950170-23-027948',
   'names': ['Netapp Inc', 'NETAPP INC'],
   'source': 'https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm',
   'cik': '1002047'}}]

In [69]:
form_info = form_info_list[0]['formInfo']
form_info

{'cusip6': '64110D',
 'formId': '0000950170-23-027948',
 'names': ['Netapp Inc', 'NETAPP INC'],
 'source': 'https://www.sec.gov/Archives/edgar/data/1002047/000095017023027948/0000950170-23-027948-index.htm',
 'cik': '1002047'}

In [70]:
cypher = """
    MERGE (f:Form {formId: $formInfoParam.formId })
      ON CREATE 
        SET f.names = $formInfoParam.names
        SET f.source = $formInfoParam.source
        SET f.cik = $formInfoParam.cik
        SET f.cusip6 = $formInfoParam.cusip6
"""

kg.query(cypher, params={'formInfoParam': form_info})

[]

In [71]:
kg.query("MATCH (f:Form) RETURN count(f) as formCount")

[{'formCount': 1}]

### Create a linked list of Chunk nodes for each section
- Start by identifying chunks from the same section

In [72]:
cypher = """
  MATCH (from_same_form:Chunk)
    WHERE from_same_form.formId = $formIdParam
  RETURN from_same_form {.formId, .f10kItem, .chunkId, .chunkSeqId } as chunkInfo
    LIMIT 10
"""

kg.query(cypher, params={'formIdParam': form_info['formId']})

[{'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0000',
   'chunkSeqId': 0}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0001',
   'chunkSeqId': 1}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0002',
   'chunkSeqId': 2}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0003',
   'chunkSeqId': 3}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0004',
   'chunkSeqId': 4}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0005',
   'chunkSeqId': 5}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-

- Order chunks by their sequence ID

In [73]:
cypher = """
  MATCH (from_same_form:Chunk)
    WHERE from_same_form.formId = $formIdParam
  RETURN from_same_form {.formId, .f10kItem, .chunkId, .chunkSeqId } as chunkInfo 
    ORDER BY from_same_form.chunkSeqId ASC
    LIMIT 10
"""

kg.query(cypher, params={'formIdParam': form_info['formId']})

[{'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item7a',
   'chunkId': '0000950170-23-027948-item7a-chunk0000',
   'chunkSeqId': 0}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item7',
   'chunkId': '0000950170-23-027948-item7-chunk0000',
   'chunkSeqId': 0}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1a',
   'chunkId': '0000950170-23-027948-item1a-chunk0000',
   'chunkSeqId': 0}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0000',
   'chunkSeqId': 0}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0001',
   'chunkSeqId': 1}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0002',
   'chunkSeqId': 2}},
 {'chunkInfo': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950

- Limit chunks to just the "Item 1" section, the organize in ascending order

In [74]:
cypher = """
  MATCH (from_same_section:Chunk)
  WHERE from_same_section.formId = $formIdParam
    AND from_same_section.f10kItem = $f10kItemParam // NEW!!!
  RETURN from_same_section { .formId, .f10kItem, .chunkId, .chunkSeqId }
    ORDER BY from_same_section.chunkSeqId ASC
    LIMIT 10
"""

kg.query(cypher, params={'formIdParam': form_info['formId'],
                         'f10kItemParam': 'item1'})

[{'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0000',
   'chunkSeqId': 0}},
 {'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0001',
   'chunkSeqId': 1}},
 {'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0002',
   'chunkSeqId': 2}},
 {'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0003',
   'chunkSeqId': 3}},
 {'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0004',
   'chunkSeqId': 4}},
 {'from_same_section': {'formId': '0000950170-23-027948',
   'f10kItem': 'item1',
   'chunkId': '0000950170-23-027948-item1-chunk0005',
   'chunkSeqId': 5}},
 {'from_same_section': {'formId': '0000950170-23-027

- Collect ordered chunks into a list

In [75]:
cypher = """
  MATCH (from_same_section:Chunk)
  WHERE from_same_section.formId = $formIdParam
    AND from_same_section.f10kItem = $f10kItemParam
  WITH from_same_section { .formId, .f10kItem, .chunkId, .chunkSeqId }
    ORDER BY from_same_section.chunkSeqId ASC
    LIMIT 10
  RETURN collect(from_same_section) // NEW!!!
"""

kg.query(cypher, params={'formIdParam': form_info['formId'],
                         'f10kItemParam': 'item1'})

[{'collect(from_same_section)': [{'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0000',
    'chunkSeqId': 0},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0001',
    'chunkSeqId': 1},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0002',
    'chunkSeqId': 2},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0003',
    'chunkSeqId': 3},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0004',
    'chunkSeqId': 4},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0005',
    'chunkSeqId': 5},
   {'formId': '0000950170-23-027948',
    'f10kItem': 'item1',
    'chunkId': '0000950170-23-027948-item1-chunk0006',
    'chunkSe

### Connect chunks to their parent form with a PART_OF relationship

In [77]:
cypher = """
  MATCH (c:Chunk), (f:Form)
    WHERE c.formId = f.formId
  MERGE (c)-[newRelationship:PART_OF]->(f)
  RETURN count(newRelationship)
"""

kg.query(cypher)

[{'count(newRelationship)': 23}]

In [79]:
cypher = """
  MATCH (first:Chunk), (f:Form)
  WHERE first.formId = f.formId
    AND first.chunkSeqId = 0
  WITH first, f
    MERGE (f)-[r:SECTION {f10kItem: first.f10kItem}]->(first)
  RETURN count(r)
"""

kg.query(cypher)

[{'count(r)': 4}]

In [80]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Movie {title: STRING, taglineEmbedding: LIST, released: INTEGER, tagline: STRING}
Person {born: INTEGER, name: STRING}
Classification {name: STRING, ers_2025: STRING, family: STRING, ers_2015: STRING}
Chunk {textEmbedding: LIST, f10kItem: STRING, chunkSeqId: INTEGER, text: STRING, cik: STRING, cusip6: STRING, names: LIST, formId: STRING, source: STRING, chunkId: STRING}
Form {cik: STRING, cusip6: STRING, names: LIST, formId: STRING, source: STRING}
Relationship properties:
ACTED_IN {roles: LIST}
REVIEWED {summary: STRING, rating: INTEGER}
SECTION {f10kItem: STRING}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person)-[:REVIEWED]->(:Movie)
(:Chunk)-[:PART_OF]->(:Form)
(:Form)-[:SECTION]->(:Chunk)


#### Create a SECTION relationship on first chunk of each section

### Add a NEXT relationship between subsequent chunks
- Use the `apoc.nodes.link` function from Neo4j to link ordered list of `Chunk` nodes with a `NEXT` relationship
- Do this for just the "Item 1" section to start

In [86]:
cypher = """
  MATCH (from_same_section:Chunk)
  WHERE from_same_section.formId = $formIdParam
    AND from_same_section.f10kItem = $f10kItemParam
  WITH from_same_section
    ORDER BY from_same_section.chunkSeqId ASC
  WITH collect(from_same_section) as section_chunk_list
    CALL apoc.nodes.link(
        section_chunk_list,
        "NEXT",
        {avoidDuplicates: true}
    )  // NEW!!!
  RETURN size(section_chunk_list)
"""

kg.query(cypher, params={'formIdParam': form_info['formId'],
                         'f10kItemParam': 'item1'})

[{'size(section_chunk_list)': 20}]

### Example cypher queries
- Return the first chunk of the Item 1 section

In [87]:
cypher = """
  MATCH (f:Form)-[r:SECTION]->(first:Chunk)
    WHERE f.formId = $formIdParam
        AND r.f10kItem = $f10kItemParam
  RETURN first.chunkId as chunkId, first.text as text
"""

first_chunk_info = kg.query(cypher, params={
    'formIdParam': form_info['formId'],
    'f10kItemParam': 'item1'
})[0]

first_chunk_info

{'chunkId': '0000950170-23-027948-item1-chunk0000',
 'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastruct

- Get the second chunk of the Item 1 section

In [88]:
cypher = """
  MATCH (first:Chunk)-[:NEXT]->(nextChunk:Chunk)
    WHERE first.chunkId = $chunkIdParam
  RETURN nextChunk.chunkId as chunkId, nextChunk.text as text
"""

next_chunk_info = kg.query(cypher, params={
    'chunkIdParam': first_chunk_info['chunkId']
})[0]

next_chunk_info

{'chunkId': '0000950170-23-027948-item1-chunk0001',
 'text': "•\nFlexibility and consistency: NetApp makes moving data and applications between environments seamless through a common storage foundation across on-premises and multicloud environments.\n\n\n•\nCyber resilience: NetApp unifies monitoring, data protection, security, governance, and compliance for total cyber resilience - with consistency and automation across environments. \n\n\n•\nContinuous operations: NetApp uses AI-driven automation for continuous optimization to service applications and store stateless and stateful applications at the lowest possible costs.\n\n\n•\nSustainability: NetApp has industry-leading tools to audit consumption, locate waste, and set guardrails to stop overprovisioning.\n\n\nProduct, Solutions and Services Portfolio\n \n\n\nNetApp's portfolio of cloud services and storage infrastructure is powered by intelligent data management software. Our operations are organized into two segments: Hybrid Clo

- Return a window of three chunks

In [89]:
cypher = """
    MATCH (c1:Chunk)-[:NEXT]->(c2:Chunk)-[:NEXT]->(c3:Chunk)
        WHERE c2.chunkId = $chunkIdParam
    RETURN c1.chunkId, c2.chunkId, c3.chunkId
    """

kg.query(cypher,
         params={'chunkIdParam': next_chunk_info['chunkId']})

[{'c1.chunkId': '0000950170-23-027948-item1-chunk0000',
  'c2.chunkId': '0000950170-23-027948-item1-chunk0001',
  'c3.chunkId': '0000950170-23-027948-item1-chunk0002'}]

### Information is stored in the structure of a graph
- Matched patterns of nodes and relationships in a graph are called **paths**
- The length of a path is equal to the number of relationships in the path
- Paths can be captured as variables and used elsewhere in queries

In [90]:
cypher = """
    MATCH window = (c1:Chunk)-[:NEXT]->(c2:Chunk)-[:NEXT]->(c3:Chunk)
        WHERE c1.chunkId = $chunkIdParam
    RETURN length(window) as windowPathLength
    """

kg.query(cypher,
         params={'chunkIdParam': next_chunk_info['chunkId']})

[{'windowPathLength': 2}]

### Finding variable length windows
- A pattern match will fail if the relationship doesn't exist in the graph
- For example, the first chunk in a section has no preceding chunk, so the next query won't return anything

In [91]:
cypher = """
    MATCH window=(c1:Chunk)-[:NEXT]->(c2:Chunk)-[:NEXT]->(c3:Chunk)
        WHERE c2.chunkId = $chunkIdParam
    RETURN nodes(window) as chunkList
    """
# pull the chunk ID from the first
kg.query(cypher,
         params={'chunkIdParam': first_chunk_info['chunkId']})

[]

- Modify `NEXT` relationship to have variable length

In [92]:
cypher = """
  MATCH window=
      (:Chunk)-[:NEXT*0..1]->(c:Chunk)-[:NEXT*0..1]->(:Chunk)
    WHERE c.chunkId = $chunkIdParam
  RETURN length(window)
  """

kg.query(cypher,
         params={'chunkIdParam': first_chunk_info['chunkId']})

[{'length(window)': 0}, {'length(window)': 1}]

- Retrieve only the longest path

In [93]:
cypher = """
  MATCH window=
      (:Chunk)-[:NEXT*0..1]->(c:Chunk)-[:NEXT*0..1]->(:Chunk)
    WHERE c.chunkId = $chunkIdParam
  WITH window as longestChunkWindow
      ORDER BY length(window) DESC LIMIT 1
  RETURN length(longestChunkWindow)
  """

kg.query(cypher,
         params={'chunkIdParam': first_chunk_info['chunkId']})

[{'length(longestChunkWindow)': 1}]

## Lesson 6: Expanding the SEC Knowledge Graph

### Read the collection of Form 13s
- Investment management firms must report on their investments in companies to the SEC by filing a document called **Form 13**
- You'll load a collection of Form 13 for managers that have invested in NetApp
- You can check out the CSV file by navigating to the data directory using the File menu at the top of the notebook

In [1]:
import csv

all_form13s = []

with open('./data/form13.csv', mode='r') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader: # each row will be a dictionary
        all_form13s.append(row)

- Look at the contents of the first 5 Form 13s

In [7]:
all_form13s[0:5]

[{'source': 'https://sec.gov/Archives/edgar/data/1000275/0001140361-23-039575.txt',
  'managerCik': '1000275',
  'managerAddress': 'ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5',
  'managerName': 'Royal Bank of Canada',
  'reportCalendarOrQuarter': '2023-06-30',
  'cusip6': '64110D',
  'cusip': '64110D104',
  'companyName': 'NETAPP INC',
  'value': '64395000000.0',
  'shares': '842850'},
 {'source': 'https://sec.gov/Archives/edgar/data/1002784/0001387131-23-009542.txt',
  'managerCik': '1002784',
  'managerAddress': '1875 Lawrence Street, Suite 300, Denver, CO, 80202-1805',
  'managerName': 'SHELTON CAPITAL MANAGEMENT',
  'reportCalendarOrQuarter': '2023-06-30',
  'cusip6': '64110D',
  'cusip': '64110D104',
  'companyName': 'NETAPP INC',
  'value': '2989085000.0',
  'shares': '39124'},
 {'source': 'https://sec.gov/Archives/edgar/data/1007280/0001007280-23-000008.txt',
  'managerCik': '1007280',
  'managerAddress': '277 E TOWN ST, COLUMBUS, OH, 43215',
  'managerName': 'PUBLIC 

In [9]:
len(all_form13s)

561

### Create company nodes in the graph
- Use the companies identified in the Form 13s to create `Company` nodes
- For now, there is only one company - NetApp

In [26]:
# work with just the first form fow now
first_form13 = all_form13s[0]

cypher = """
MERGE (com:Company {cusip6: $cusip6})
  ON CREATE
    SET com.companyName = $companyName,
        com.cusip = $cusip
"""

kg.query(cypher, params={
    'cusip6':first_form13['cusip6'],
    'companyName':first_form13['companyName'],
    'cusip':first_form13['cusip']
})

[]

In [29]:
print(json.dumps(first_form13, indent=2))

{
  "source": "https://sec.gov/Archives/edgar/data/1000275/0001140361-23-039575.txt",
  "managerCik": "1000275",
  "managerAddress": "ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5",
  "managerName": "Royal Bank of Canada",
  "reportCalendarOrQuarter": "2023-06-30",
  "cusip6": "64110D",
  "cusip": "64110D104",
  "companyName": "NETAPP INC",
  "value": "64395000000.0",
  "shares": "842850"
}


In [13]:
cypher = """
MATCH (com:Company)
RETURN com LIMIT 1
"""

kg.query(cypher)

[{'com': {'cusip': '64110D104',
   'companyName': 'NETAPP INC',
   'cusip6': '64110D'}}]

- Check the company name

In [14]:
cypher = """
  MATCH (com:Company), (form:Form)
    WHERE com.cusip6 = form.cusip6
  RETURN com.companyName, form.names
"""

kg.query(cypher)

[{'com.companyName': 'NETAPP INC', 'form.names': ['Netapp Inc', 'NETAPP INC']}]

- Set  the company name to match Form 10-K (in this case, it already matched, but here's the query)

In [15]:
cypher = """
  MATCH (com:Company), (form:Form)
    WHERE com.cusip6 = form.cusip6
  SET com.names = form.names
"""

kg.query(cypher)

[]

- Create a `FILED` relationship between the company and the Form-10K node

In [17]:
kg.query("""
  MATCH (com:Company), (form:Form)
    WHERE com.cusip6 = form.cusip6
  MERGE (com)-[:FILED]->(form)
""")

[]

### Create manager nodes
- Create a `manager` node for companies that have filed a Form 13 to report their investment in NetApp
- Start with the single manager who filed the first Form 13 in the list

In [18]:
cypher = """
  MERGE (mgr:Manager {managerCik: $managerParam.managerCik})
    ON CREATE
        SET mgr.managerName = $managerParam.managerName,
            mgr.managerAddress = $managerParam.managerAddress
"""

kg.query(cypher, params={'managerParam': first_form13})

[]

In [19]:
kg.query("""
  MATCH (mgr:Manager)
  RETURN mgr LIMIT 1
""")

[{'mgr': {'managerCik': '1000275',
   'managerAddress': 'ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5',
   'managerName': 'Royal Bank of Canada'}}]

- Create a uniquness constraint to avoid duplicate managers

In [20]:
kg.query("""
CREATE CONSTRAINT unique_manager
  IF NOT EXISTS
  FOR (n:Manager)
  REQUIRE n.managerCik IS UNIQUE
""")

[]

- Create a fulltext index of manager names to enable text search

In [21]:
kg.query("""
CREATE FULLTEXT INDEX fullTextManagerNames
  IF NOT EXISTS
  FOR (mgr:Manager)
  ON EACH [mgr.managerName]
""")


[]

In [22]:
kg.query("""
  CALL db.index.fulltext.queryNodes("fullTextManagerNames",
      "royal bank") YIELD node, score
  RETURN node.managerName, score
""")

[{'node.managerName': 'Royal Bank of Canada', 'score': 0.2615291476249695}]

- Create nodes for all companies that filed a Form 13

In [23]:
cypher = """
  MERGE (mgr:Manager {managerCik: $managerParam.managerCik})
    ON CREATE
        SET mgr.managerName = $managerParam.managerName,
            mgr.managerAddress = $managerParam.managerAddress
"""
# loop through all Form 13s
for form13 in all_form13s:
    kg.query(cypher, params={'managerParam': form13 })

In [24]:
kg.query("""
    MATCH (mgr:Manager)
    RETURN count(mgr)
""")

[{'count(mgr)': 561}]

### Create relationships between managers and companies
- Match companies with managers based on data in the Form 13
- Create an `OWNS_STOCK_IN` relationship between the manager and the company
- Start with the single manager who filed the first Form 13 in the list

In [25]:
# {
#     "source": "https://sec.gov/Archives/edgar/data/1000275/0001140361-23-039575.txt",
#     "managerCik": "1000275",
#     "managerAddress": "ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5",
#     "managerName": "Royal Bank of Canada",
#     "reportCalendarOrQuarter": "2023-06-30",
#     "cusip6": "64110D",
#     "cusip": "64110D104",
#     "companyName": "NETAPP INC",
#     "value": "64395000000.0",
#     "shares": "842850"
# }

cypher = """
  MATCH (mgr:Manager {managerCik: $investmentParam.managerCik}),
        (com:Company {cusip6: $investmentParam.cusip6})
  RETURN mgr.managerName, com.companyName, $investmentParam as investment
"""

kg.query(cypher, params={
    'investmentParam': first_form13
})

[{'mgr.managerName': 'Royal Bank of Canada',
  'com.companyName': 'NETAPP INC',
  'investment': {'managerCik': '1000275',
   'shares': '842850',
   'cusip': '64110D104',
   'managerAddress': 'ROYAL BANK PLAZA, 200 BAY STREET, TORONTO, A6, M5J2J5',
   'reportCalendarOrQuarter': '2023-06-30',
   'companyName': 'NETAPP INC',
   'cusip6': '64110D',
   'source': 'https://sec.gov/Archives/edgar/data/1000275/0001140361-23-039575.txt',
   'managerName': 'Royal Bank of Canada',
   'value': '64395000000.0'}}]

In [30]:
cypher = """
MATCH (mgr:Manager {managerCik: $ownsParam.managerCik}),
        (com:Company {cusip6: $ownsParam.cusip6})
MERGE (mgr)-[owns:OWNS_STOCK_IN {
    reportCalendarOrQuarter: $ownsParam.reportCalendarOrQuarter
}]->(com)
ON CREATE
    SET owns.value  = toFloat($ownsParam.value),
        owns.shares = toInteger($ownsParam.shares)
RETURN mgr.managerName, owns.reportCalendarOrQuarter, com.companyName
"""

kg.query(cypher, params={ 'ownsParam': first_form13 })

[{'mgr.managerName': 'Royal Bank of Canada',
  'owns.reportCalendarOrQuarter': '2023-06-30',
  'com.companyName': 'NETAPP INC'}]

In [31]:
kg.query("""
MATCH (mgr:Manager {managerCik: $ownsParam.managerCik})
-[owns:OWNS_STOCK_IN]->
        (com:Company {cusip6: $ownsParam.cusip6})
RETURN owns { .shares, .value }
""", params={ 'ownsParam': first_form13 })

[{'owns': {'shares': 842850, 'value': 64395000000.0}}]

- Create relationships between all of the managers who filed Form 13s and the company

In [32]:
cypher = """
MATCH (mgr:Manager {managerCik: $ownsParam.managerCik}),
        (com:Company {cusip6: $ownsParam.cusip6})
MERGE (mgr)-[owns:OWNS_STOCK_IN {
    reportCalendarOrQuarter: $ownsParam.reportCalendarOrQuarter
    }]->(com)
  ON CREATE
    SET owns.value  = toFloat($ownsParam.value),
        owns.shares = toInteger($ownsParam.shares)
"""

#loop through all Form 13s
for form13 in all_form13s:
    kg.query(cypher, params={'ownsParam': form13 })

In [33]:
cypher = """
  MATCH (:Manager)-[owns:OWNS_STOCK_IN]->(:Company)
  RETURN count(owns) as investments
"""

kg.query(cypher)

[{'investments': 561}]

In [41]:
kg.refresh_schema()
print(textwrap.fill(kg.schema, 60))

Node properties: Movie {title: STRING, taglineEmbedding:
LIST, released: INTEGER, tagline: STRING} Person {born:
INTEGER, name: STRING} Classification {name: STRING,
ers_2025: STRING, family: STRING, ers_2015: STRING} Chunk
{textEmbedding: LIST, f10kItem: STRING, chunkSeqId: INTEGER,
text: STRING, cik: STRING, cusip6: STRING, names: LIST,
formId: STRING, source: STRING, chunkId: STRING} Form {cik:
STRING, cusip6: STRING, names: LIST, formId: STRING, source:
STRING} Company {cusip: STRING, names: LIST, cusip6: STRING,
companyName: STRING} Manager {managerName: STRING,
managerCik: STRING, managerAddress: STRING} Relationship
properties: ACTED_IN {roles: LIST} REVIEWED {summary:
STRING, rating: INTEGER} SECTION {f10kItem: STRING}
OWNS_STOCK_IN {shares: INTEGER, reportCalendarOrQuarter:
STRING, value: FLOAT} The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie) (:Person)-[:WROTE]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person